In [ ]:
import yaml
import umap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from src.features.plot import plot_umap

In [ ]:
with open("../configs/default.yaml") as f:
    cfg = yaml.safe_load(f)

name = (
    f"model_{cfg['model']['type']}"
    f"_latent{cfg['model']['latent']}"
    f"_ch{'_'.join(str(c) for c in cfg['model']['channels'])}"
    f"_beta{cfg['train']['beta']}"
    f"_lr{cfg['optimizer']['lr']}"
    f"_epoch{cfg['train']['epochs']}"
    f"_act{cfg['model']['activation']}"
    f"_kern{cfg['model']['kernel']}"
    f"_stride{cfg['model']['stride']}"
    f"_pad{cfg['model']['padding']}"
)

inference_dir = Path(cfg["output"]["inference_dir"]) / name

train = np.load(inference_dir / "train.npz")
val   = np.load(inference_dir / "val.npz")
kaon  = np.load(inference_dir / "kaon.npz")

train_latents = train["latents"]
train_recon   = train["recon"]
train_re      = train["re"]

val_latents   = val["latents"]
val_recon     = val["recon"]
val_re        = val["re"]

kaon_latents  = kaon["latents"]
kaon_recon    = kaon["recon"]
kaon_re       = kaon["re"]

In [ ]:
features = pd.read_pickle('/Volumes/easystore/proton-kaon/features/features.pkl')

In [ ]:
index = np.load('/Volumes/easystore/proton-kaon/training/split_p.npz')

In [ ]:
features.columns

In [ ]:
train_features = features[features['particle_type'] == 'proton'].iloc[index['train_idx']]
val_features = features[features['particle_type'] == 'proton'].iloc[index['val_idx']]
kaon_features = features[features['particle_type'] == 'kaon']

In [ ]:
(train_latents.shape, train_features.shape), (val_latents.shape, val_features.shape), (kaon_latents.shape, kaon_features.shape)

In [ ]:
all_latents = np.vstack([train_latents, val_latents, kaon_latents])
reducer = umap.UMAP(n_neighbors=30, min_dist=0.1)
reducer.fit(all_latents)

train_umap = reducer.transform(train_latents)
val_umap = reducer.transform(val_latents)
kaon_umap = reducer.transform(kaon_latents)

In [ ]:
fig, axes = plot_umap(train_umap, train_features, val_umap, val_features, kaon_umap, kaon_features, 'total_adc')
plt.show()